# SRL Training with Repository Functions

This notebook demonstrates how to use the **Supervised-Reinforcement-Learning** repository's actual functions for training. Unlike the basic tutorial, this notebook imports and uses the real modules:

- `srl_reward_function.py` - SRLRewardFunction class with dynamic sampling
- `rlvr_reward_function.py` - RLVRRewardFunction for final answer verification
- `sdk_to_srl.py` - Data processing utilities
- `train_srl.py` - Dataset loading and reward function creation

## Training Pipeline
```
Stage 1: SRL → Learn step-by-step reasoning (this notebook)
Stage 2: RLVR → Learn correct final answers (bonus section)
```

## 1. Setup and Installation

In [ ]:
# For Kaggle/Colab: Install dependencies
# %%capture
# !pip install unsloth trl datasets cdifflib
# !pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import os
import sys

# Add the srl module to path (adjust path as needed)
SRL_PATH = os.path.expanduser("~/Supervised-Reinforcement-Learning/srl")
if SRL_PATH not in sys.path:
    sys.path.insert(0, SRL_PATH)

# For Kaggle: use the input path
# SRL_PATH = "/kaggle/input/your-dataset-name/srl"
# sys.path.insert(0, SRL_PATH)

print(f"SRL module path: {SRL_PATH}")

## 2. Import Repository Functions

Now we import the actual functions from the repository modules.

In [ ]:
# Import reward function classes
from srl_reward_function import SRLRewardFunction, DynamicSamplingFilter
from rlvr_reward_function import RLVRRewardFunction, create_rlvr_reward_function

# Import data processing utilities
from sdk_to_srl import split_cot_into_steps, create_srl_pairs, infer_topic

# Import training utilities
from train_srl import load_srl_dataset, create_srl_reward_function, SRL_INSTRUCTION

print("✓ All repository modules imported successfully!")

## 3. Explore the SRL Reward Function

The `SRLRewardFunction` class computes sequence similarity between generated steps and expert actions.

In [ ]:
# Initialize the reward function
srl_reward = SRLRewardFunction(
    format_check=False,           # Don't enforce strict format
    min_similarity=0.0,           # Minimum similarity floor
    penalty_for_format_error=-1.0,
    use_dynamic_filter=True,      # Enable dynamic sampling
    variance_threshold=0.01       # Filter low-variance samples
)

print("SRL Reward Function initialized with:")
print(f"  - Format check: {srl_reward.format_check}")
print(f"  - Dynamic filter: {srl_reward.use_dynamic_filter}")
print(f"  - Variance threshold: {srl_reward.dynamic_filter.variance_threshold if srl_reward.dynamic_filter else 'N/A'}")

In [ ]:
# Test the reward function with example step comparisons
test_cases = [
    {
        "generated": "Step 2: Subtract 5 from both sides to get x = 7.",
        "expert": "Step 2: Subtract 5 from both sides. x + 5 - 5 = 12 - 5, so x = 7."
    },
    {
        "generated": "Step 2: x = 7",
        "expert": "Step 2: Subtract 5 from both sides. x + 5 - 5 = 12 - 5, so x = 7."
    },
    {
        "generated": "Step 2: Add 5 to both sides.",  # Wrong approach
        "expert": "Step 2: Subtract 5 from both sides."
    }
]

print("Testing SRL Reward Function:\n")
for i, case in enumerate(test_cases, 1):
    reward = srl_reward(case["generated"], case["expert"])
    details = srl_reward.get_similarity_details(case["generated"], case["expert"])
    
    print(f"Test {i}:")
    print(f"  Generated: {case['generated'][:50]}...")
    print(f"  Expert:    {case['expert'][:50]}...")
    print(f"  Reward:    {reward:.4f}")
    print(f"  Similarity: {details['similarity']:.4f}")
    print()

## 4. Understand Dynamic Sampling Filter

The `DynamicSamplingFilter` implements Section 4.2 of the SRL paper - filtering out "easy" samples where all rollouts get similar rewards.

In [ ]:
# Initialize dynamic sampling filter
dynamic_filter = DynamicSamplingFilter(variance_threshold=0.01)

# Test with different reward distributions
test_reward_sets = [
    [0.95, 0.94, 0.96, 0.95],  # Low variance - all similar (should filter)
    [0.9, 0.5, 0.8, 0.3],      # High variance - diverse (should keep)
    [0.0, 0.0, 0.0, 0.0],      # All zeros (should filter)
    [1.0, 0.0, 0.5, 0.8],      # Mixed (should keep)
]

print("Dynamic Sampling Filter Test:\n")
for rewards in test_reward_sets:
    keep = dynamic_filter.should_keep_sample(rewards)
    import statistics
    var = statistics.variance(rewards) if len(rewards) > 1 else 0
    print(f"Rewards: {rewards}")
    print(f"  Variance: {var:.6f}")
    print(f"  Keep sample: {keep}")
    print()

## 5. Data Processing with sdk_to_srl

Convert Chain-of-Thought data into SRL step-wise training pairs.

In [ ]:
# Example Chain-of-Thought solution
example_cot = """Step 1: Identify the equation x + 5 = 12.
Step 2: Subtract 5 from both sides of the equation.
Step 3: Simplify to get x = 12 - 5.
Step 4: Calculate x = 7.
Final Answer: x = 7"""

example_question = "Solve for x: x + 5 = 12"

# Split CoT into steps
steps, answer_line = split_cot_into_steps(example_cot)

print("Split Chain-of-Thought into steps:\n")
print(f"Number of steps: {len(steps)}")
for i, step in enumerate(steps):
    print(f"  [{i}] {step}")
print(f"\nAnswer line: {answer_line}")

In [ ]:
# Create SRL training pairs from the steps
srl_pairs = create_srl_pairs(
    question=example_question,
    steps=steps,
    answer_line=answer_line,
    topic=infer_topic(example_question)
)

print(f"Created {len(srl_pairs)} SRL training pairs:\n")
for i, pair in enumerate(srl_pairs):
    print(f"Pair {i+1}:")
    print(f"  Input Prompt (last 80 chars): ...{pair['input_prompt'][-80:]}")
    print(f"  Expert Action: {pair['expert_action']}")
    print(f"  Topic: {pair.get('topic', 'N/A')}")
    print()

## 6. Load Model with Unsloth + vLLM

Load a quantized model with Unsloth for memory-efficient training.

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL

# Patch TRL for Unsloth compatibility
PatchFastRL("GRPO", FastLanguageModel)

# Model configuration
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
GPU_MEMORY_UTIL = 0.6

# Load model with vLLM for fast inference
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=True,         # Enable vLLM with sleep mode
    gpu_memory_utilization=GPU_MEMORY_UTIL,
)

print(f"✓ Model loaded: {MODEL_NAME}")

In [ ]:
# Add LoRA adapters for efficient fine-tuning
LORA_RANK = 16

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing="unsloth",
)

print(f"✓ LoRA adapters added (rank={LORA_RANK})")

## 7. Load SRL Dataset Using Repository Function

Use the `load_srl_dataset` function from `train_srl.py`.

In [ ]:
# Option 1: Load from existing JSONL file
# DATA_PATH = os.path.join(SRL_PATH, "srl_datasets/srl_train.jsonl")
# train_dataset = load_srl_dataset(
#     data_path=DATA_PATH,
#     tokenizer=tokenizer,
#     use_instruction=True
# )

# Option 2: Create a sample dataset for demonstration
from datasets import Dataset

# Sample SRL training data
sample_data = [
    {
        "input_prompt": "Problem: If x + 5 = 12, what is x?\n\nStep 1: Identify the equation.",
        "expert_action": "Step 2: Subtract 5 from both sides to isolate x."
    },
    {
        "input_prompt": "Problem: If x + 5 = 12, what is x?\n\nStep 1: Identify the equation.\nStep 2: Subtract 5 from both sides.",
        "expert_action": "Step 3: x = 12 - 5 = 7."
    },
    {
        "input_prompt": "Problem: Calculate 15% of 80.",
        "expert_action": "Step 1: Convert 15% to decimal: 15/100 = 0.15"
    },
    {
        "input_prompt": "Problem: Calculate 15% of 80.\n\nStep 1: Convert 15% to decimal: 0.15",
        "expert_action": "Step 2: Multiply 0.15 × 80 = 12."
    },
    {
        "input_prompt": "Problem: Find the area of a triangle with base 6 and height 4.",
        "expert_action": "Step 1: Recall the formula: Area = (1/2) × base × height."
    },
    {
        "input_prompt": "Problem: Find the area of a triangle with base 6 and height 4.\n\nStep 1: Recall the formula: Area = (1/2) × base × height.",
        "expert_action": "Step 2: Substitute values: Area = (1/2) × 6 × 4 = 12 square units."
    },
]

# Apply chat template to prompts
def format_prompt(example):
    messages = [
        {"role": "system", "content": SRL_INSTRUCTION},
        {"role": "user", "content": example["input_prompt"]}
    ]
    return {
        "prompt": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
        "expert_action": example["expert_action"]
    }

train_dataset = Dataset.from_list([format_prompt(s) for s in sample_data])

print(f"✓ Dataset loaded: {len(train_dataset)} samples")
print(f"\nSample prompt preview:")
print(train_dataset[0]["prompt"][:300] + "...")

## 8. Create TRL-Compatible Reward Function

Use `create_srl_reward_function` from the repository to create a GRPOTrainer-compatible reward function.

In [ ]:
# Create the TRL-compatible reward function
reward_fn = create_srl_reward_function(
    format_check=False,         # Don't enforce strict step format
    use_dynamic_filter=True     # Enable dynamic sampling
)

print("✓ TRL-compatible reward function created")
print(f"\nReward function signature: {reward_fn.__code__.co_varnames[:5]}")

In [ ]:
# Test the reward function with mock TRL-style inputs
mock_completions = [
    "Step 2: Subtract 5 from both sides. x = 12 - 5 = 7.",
    "Step 2: x equals 7.",
    "Step 2: Add 5 to get the answer.",
    "Invalid output without step format"
]
mock_expert_actions = [
    "Step 2: Subtract 5 from both sides to isolate x."
] * 4

# Compute rewards (TRL passes these as kwargs)
rewards = reward_fn(
    completions=mock_completions,
    prompts=[None] * 4,  # Unused
    expert_action=mock_expert_actions
)

print("Reward Function Test Results:\n")
for comp, reward in zip(mock_completions, rewards):
    print(f"  Completion: {comp[:50]}...")
    print(f"  Reward: {reward:.4f}\n")

## 9. Configure GRPOTrainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = "./srl_training_output"

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    
    # Training parameters
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    
    # GRPO-specific parameters
    num_generations=4,            # K rollouts per prompt
    max_completion_length=256,    # Short for step-level generation
    temperature=1.0,
    
    # Memory optimization
    bf16=True,
    gradient_checkpointing=True,
    
    # vLLM configuration
    use_vllm=True,
    vllm_gpu_memory_utilization=GPU_MEMORY_UTIL,
    
    # Logging
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

print("✓ GRPOConfig created")
print(f"\nKey settings:")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")
print(f"  - Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  - Num generations (K): {training_args.num_generations}")
print(f"  - Max completion length: {training_args.max_completion_length}")

## 10. Initialize and Run Training

In [ ]:
# Initialize the GRPOTrainer
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=reward_fn,
    tokenizer=tokenizer,
)

print("✓ GRPOTrainer initialized")

In [ ]:
# Start training
print("Starting SRL Training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("✓ Training complete!")

## 11. Test the Trained Model

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

# Test problems
test_problems = [
    "Problem: Solve 2x + 3 = 11 for x.\n\nStep 1: Subtract 3 from both sides.",
    "Problem: What is 25% of 200?\n\nStep 1: Convert 25% to decimal: 0.25",
    "Problem: Find the perimeter of a square with side 5cm."
]

print("Testing trained model:\n")
print("=" * 60)

for problem in test_problems:
    messages = [
        {"role": "system", "content": SRL_INSTRUCTION},
        {"role": "user", "content": problem}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the generated step
    generated_step = response.split(problem)[-1].strip()
    
    print(f"Input: {problem}")
    print(f"\nGenerated Next Step: {generated_step}")
    print("=" * 60)

## 12. Bonus: RLVR Reward Function

After SRL training (Stage 1), you can use RLVR (Stage 2) to fine-tune for correct final answers.

In [ ]:
# Initialize RLVR reward function
rlvr_reward = RLVRRewardFunction(case_sensitive=False)

# Test answer extraction and verification
test_outputs = [
    ("Step 1: Analyze... Step 2: Calculate... Final Answer: A", "A"),
    ("The answer is B based on the constraints.", "B"),
    ("After checking all possibilities, Final Answer: Alice", "Alice"),
    ("I'm not sure about the answer.", "C"),  # No clear answer
]

print("RLVR Reward Function Test:\n")
for output, correct in test_outputs:
    extracted = rlvr_reward.extract_answer(output)
    reward = rlvr_reward(output, correct)
    
    print(f"Output: {output[:50]}...")
    print(f"  Correct answer: {correct}")
    print(f"  Extracted: {extracted}")
    print(f"  Reward: {reward}")
    print()

In [ ]:
# Create TRL-compatible RLVR reward function for Stage 2 training
rlvr_reward_fn = create_rlvr_reward_function()

# Test with TRL-style inputs
mock_rlvr_completions = [
    "After analysis, Final Answer: A",
    "The answer is B",
    "Final Answer: C",
]
mock_correct_answers = ["A", "A", "C"]  # First two should fail, third should pass

rlvr_rewards = rlvr_reward_fn(
    completions=mock_rlvr_completions,
    prompts=[None] * 3,
    correct_answer=mock_correct_answers
)

print("RLVR TRL Reward Function Test:\n")
for comp, correct, reward in zip(mock_rlvr_completions, mock_correct_answers, rlvr_rewards):
    print(f"  Completion: {comp}")
    print(f"  Correct: {correct} | Reward: {reward}")
    print()

## 13. Save the Model

In [ ]:
# Save the trained LoRA adapter
SAVE_PATH = "./srl_trained_model"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✓ Model saved to {SAVE_PATH}")

## Summary

This notebook demonstrated how to use the **Supervised-Reinforcement-Learning** repository functions:

| Module | Function/Class | Purpose |
|--------|---------------|--------|
| `srl_reward_function` | `SRLRewardFunction` | Sequence similarity reward |
| `srl_reward_function` | `DynamicSamplingFilter` | Filter low-variance samples |
| `rlvr_reward_function` | `RLVRRewardFunction` | Final answer verification |
| `sdk_to_srl` | `split_cot_into_steps` | Parse CoT into steps |
| `sdk_to_srl` | `create_srl_pairs` | Create training pairs |
| `train_srl` | `load_srl_dataset` | Load JSONL dataset |
| `train_srl` | `create_srl_reward_function` | TRL-compatible reward |

**Next steps:**
1. Train on your full dataset using `train_srl.py`
2. Run Stage 2 RLVR training with `train_srl_rlvr.py`
3. Evaluate with `test_model.py`